# MLOps Practical: YAML Configuration for ML Training

**Aim:** Understand YAML syntax and use a YAML config file to drive a Random Forest training run in Python.

**Idea in one line:** YAML is like a recipe card. The Python code is the cook, and the card says how many trees, how deep, and how much data to hold out. Change the card, not the cook.

## Step 1: Install libraries

In [1]:
!pip install -q pyyaml scikit-learn joblib

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## Step 2: Create project folder

In [2]:
import os
os.makedirs("YAML_MLOPS", exist_ok=True)
os.chdir("YAML_MLOPS")
print("Working directory:", os.getcwd())

Working directory: /home/claude/YAML_MLOPS


## Step 3: Create `config.yaml`

In [3]:
%%writefile config.yaml
# Project settings
project_name: YAML_MLOPS_Practical

# Model hyperparameters
model:
  algorithm: RandomForest
  n_estimators: 100
  max_depth: 5
  random_state: 42

# Training settings
training:
  test_size: 0.2

Writing config.yaml


**Syntax recap:** `key: value` pairs, 2-space indentation (never tabs) for nesting, `-` for lists, `#` for comments.

In [4]:
# Quick demo of YAML lists + nesting
import yaml
demo = '''
datasets:
  - breast_cancer
  - iris
model:
  name: RF
'''
print(yaml.safe_load(demo))

{'datasets': ['breast_cancer', 'iris'], 'model': {'name': 'RF'}}


## Step 4: Create `train.py`

In [5]:
%%writefile train.py
import yaml
import joblib
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Step 1: Load YAML configuration
with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)

print("Project:", config["project_name"])

# Step 2: Load the dataset
data = load_breast_cancer()
X = data.data
y = data.target

# Step 3: Split data using YAML settings
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=config["training"]["test_size"],
    random_state=config["model"]["random_state"]
)

# Step 4: Create model using YAML parameters
model = RandomForestClassifier(
    n_estimators=config["model"]["n_estimators"],
    max_depth=config["model"]["max_depth"],
    random_state=config["model"]["random_state"]
)

# Step 5: Train the model
model.fit(X_train, y_train)

# Step 6: Evaluate the model
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print("Algorithm:", config["model"]["algorithm"])
print("Number of trees:", config["model"]["n_estimators"])
print("Test data proportion:", config["training"]["test_size"])
print("Model Accuracy:", round(accuracy * 100, 2), "%")

# Step 7: Save the trained model
joblib.dump(model, "model.pkl")
print("Model saved successfully as model.pkl")

Writing train.py


## Step 5: Execute the program

In [6]:
!python train.py

Project: YAML_MLOPS_Practical


Algorithm: RandomForest
Number of trees: 100
Test data proportion: 0.2
Model Accuracy: 96.49 %


Model saved successfully as model.pkl


## Step 6: Observe the output and folder

In [7]:
print(sorted(os.listdir(".")))

['config.yaml', 'model.pkl', 'train.py']


In [8]:
# Verify the saved model loads and predicts
import joblib
from sklearn.datasets import load_breast_cancer
m = joblib.load("model.pkl")
X = load_breast_cancer().data
print("Loaded:", m)
print("Sample predictions:", m.predict(X[:5]))

Loaded: RandomForestClassifier(max_depth=5, random_state=42)
Sample predictions: [0 0 0 0 0]


## Step 7: Experiment with YAML parameters
Change `n_estimators` from 100 to 50 in `config.yaml` and re-run. **No change to the Python code.**

In [9]:
import yaml
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["model"]["n_estimators"] = 50
with open("config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(open("config.yaml").read())

project_name: YAML_MLOPS_Practical
model:
  algorithm: RandomForest
  n_estimators: 50
  max_depth: 5
  random_state: 42
training:
  test_size: 0.2



In [10]:
!python train.py

Project: YAML_MLOPS_Practical


Algorithm: RandomForest
Number of trees: 50
Test data proportion: 0.2
Model Accuracy: 96.49 %
Model saved successfully as model.pkl


## Bonus: compare several configs in one go

In [11]:
import yaml
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)
rows = []
for n in [10, 50, 100, 200]:
    cfg["model"]["n_estimators"] = n
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=cfg["training"]["test_size"],
        random_state=cfg["model"]["random_state"])
    clf = RandomForestClassifier(
        n_estimators=cfg["model"]["n_estimators"],
        max_depth=cfg["model"]["max_depth"],
        random_state=cfg["model"]["random_state"]).fit(Xtr, ytr)
    rows.append((n, round(accuracy_score(yte, clf.predict(Xte)) * 100, 2)))
for n, acc in rows:
    print(f"n_estimators={n:<4} accuracy={acc}%")

n_estimators=10   accuracy=95.61%
n_estimators=50   accuracy=96.49%
n_estimators=100  accuracy=96.49%
n_estimators=200  accuracy=96.49%


## Why YAML matters in MLOps
1. **Configuration management:** parameters and paths live in one file.
2. **Reproducibility:** easy to record and reuse experiment settings.
3. **Automation:** CI/CD tools (GitHub Actions, AWS CodeBuild) are defined in YAML.
4. **Maintainability:** tweak parameters without touching code.
5. **Deployment:** Docker Compose and Kubernetes are configured with YAML.

## Result
The YAML configuration was created and loaded into Python. Its parameters trained and evaluated a Random Forest classifier, and the trained model was saved as `model.pkl`. Changing `n_estimators` in the YAML altered the model without any code change.